# Set-up

In [1]:
import os

# @markdown **Crecerelle:** Install the library from GitHub
!pip install --upgrade --no-cache-dir \
    "git+https://github.com/Hollfelder-Lab/crecerelle.git" \
    "pandas==2.2.3" \
    "scvi-tools==1.4.3" \
    "scverse-misc[settings]==0.1.5" \
    "pydantic-settings" \
    "python-dotenv"

  Cloning https://github.com/Hollfelder-Lab/crecerelle.git to /tmp/pip-req-build-65xj_3sj
  Running command git clone --filter=blob:none --quiet https://github.com/Hollfelder-Lab/crecerelle.git /tmp/pip-req-build-65xj_3sj
  Resolved https://github.com/Hollfelder-Lab/crecerelle.git to commit 33aa181c44a4de0dd8a2baba283e24a61f4e00dd
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 205.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of scanpy to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of anndata to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of scanpy[skmisc] to determine which ver

In [ ]:
# @markdown **Directory structure:** Set-up the directory structure to store the results of Crecerelle. We recommend connecting to your Google Drive. Otherwise, the directories will be created only temporarily in Colab. If you have already set-up the directory structure from a previous analysis, the existing directories will not be overwritten but only the new ones added.

connect_google_drive = True # @param {type:"boolean"}

if connect_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

    root_directory = "/content/drive/My Drive"
else:
    root_directory = "/content"





Mounted at /content/drive


The following directory structure is then created:
```text
root_directory/
└── crecerelle_results/
    ├── data/
    │   └── <dataset_name>/  *(optional)*
    ├── figures/
    │   └── <dataset_name>/  *(optional)*
    └── models/
        ├── TRVI/
        ├── tuVI/
        └── scVI/

In [ ]:
from crecerelle.utils import setup_crecerelle

# @markdown **Dataset name:** Define the dataset name
dataset_name = "tabulaMuris" # @param

# Set-up the directory structure
setup_crecerelle(root_directory, dataset_name=dataset_name)

# @markdown The working directory is then set to "./crecerelle_results"
if os.path.basename(os.path.normpath(os.getcwd())) != "crecerelle_results":
    os.chdir("drive/My Drive/crecerelle_results")

# Print the working directory
print(f"The working directory is set to: {os.getcwd()}")

Skipped (already exists): /content/drive/My Drive/crecerelle_results
Skipped (already exists): /content/drive/My Drive/crecerelle_results/models
Skipped (already exists): /content/drive/My Drive/crecerelle_results/data
Skipped (already exists): /content/drive/My Drive/crecerelle_results/figures
Skipped (already exists): /content/drive/My Drive/crecerelle_results/models/scVI
Skipped (already exists): /content/drive/My Drive/crecerelle_results/models/tuVI
Skipped (already exists): /content/drive/My Drive/crecerelle_results/models/TRVI
Skipped (already exists): /content/drive/My Drive/crecerelle_results/data/tabulaMuris
Skipped (already exists): /content/drive/My Drive/crecerelle_results/figures/tabulaMuris
The working directory is set to: /content/drive/My Drive/crecerelle_results


In [ ]:
# @markdown **Imports:** modules from PyTorch, Numpy, Scanpy amongst others are imported together with the crecerelle modules necessary for training of tuVI
# General import
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from umap import UMAP
import scib

# Imports from crecerelle
from crecerelle.training_utils import train_VAE, save_model_checkpoint
from crecerelle.cell import TABULA_MURIS_CELL_TYPES, MOUSE_CORTEX_BICCN_CELL_TYPES, TABULA_MURIS_TISSUE_ORGAN_SYSTEM_DICT, TABULA_MURIS_CELL_TYPE_ABBREVIATION_DICT
from crecerelle.utils import GeneExpressionDataset, TranscriptUsageDataset, intron_names_2_integers

# Settings for plotting
plt.rcParams.update({
    'font.size': 16,              # General font size
    'axes.labelsize': 16,         # Font size for x and y labels
    'axes.titlesize': 16,         # Font size for subplot titles
    'xtick.labelsize': 16,        # Font size for x-axis tick labels
    'ytick.labelsize': 16,        # Font size for y-axis tick labels
    'legend.fontsize': 16,        # Font size for legend labels
    #'font.family': 'sans-serif',
    #'font.sans-serif': 'Roboto',
    # 'figure.titlesize': 10,    # Font size for the overall figure title (if you use it)
})

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# @markdown **Specify settings:** These are the general settings for keys to be defined. They are stored in a dictionary called settings. The dataset_name key is already saved in it. The current settings contain the default version for the Tabula Muris dataset.

# @markdown Key of Anndata object specifiying the cell type
settings_cell_type_key = "cell_ontology_class" # @param

# @markdown  Cell types of dataset given
settings_cell_types = TABULA_MURIS_CELL_TYPES # @param

# @markdown Abbreviations of cell types
settings_cell_type_abbreviations = TABULA_MURIS_CELL_TYPE_ABBREVIATION_DICT # @param

# @markdown Organisational entity of cell groups (e.g. tissue)
settings_cell_type_groups_key = "tissue" # @param

# @markdown Name of gene annotation file
settings_gene_annotation = "Mus_musculus.GRCm38.102.chr.gtf.gz" # @param

# @markdown Number of highly variable genes selected (HVGs)
settings_num_hvg = 3000 # @param

# @markdown Data partition used (recommended: "full") for tasks
settings_data_type = "full" # @param

# @markdown Key of Anndata object specifying the data partition (e.g. "train", "val", "test)
settings_data_partition_key = "data_partition" # @param

# @markdown Filter data to cell types with a minimum count
settings_filter_cell_types = True # @param {type:"boolean"}

# @markdown Minimum count threshold for filtering if filter_cell_types is True
settings_min_count_threshold = 100 # @param {type:"integer"}

settings = {
    "dataset_name": dataset_name, # Name of dataset
    "cell_type_key": settings_cell_type_key, # Key of Anndata object specifiying the cell type
    "cell_types": settings_cell_types, # Cell types of dataset given
    "cell_type_abbreviations": settings_cell_type_abbreviations, # Abbreviations of cell types
    "cell_type_groups_key": settings_cell_type_groups_key, # Organisational entity of cell groups (e.g. tissue)
    "gene_annotation": settings_gene_annotation,
    "num_hvg": settings_num_hvg, # Number of highly variable genes selected (HVGs)
    "data_type": settings_data_type, # Data partition used (recommended: "full") for tasks
    "data_partition_key": settings_data_partition_key,
    "filter_cell_types": settings_filter_cell_types, # Filter data to cell types with a minimum count
    "min_count_threshold": settings_min_count_threshold, # Minimum count threshold for filtering if filter_cell_types is True
}

In [ ]:
# @markdown All data files must be stored here and the resulting new data will also be stored here.
path2data = "./data/" + settings["dataset_name"] + "/"

print(f"The data files will be stored in: {path2data}")

The data files will be stored in: ./data/tabulaMuris/


# Loading of data

Ensure the Anndata objects of the gene expression and transcript usage data are uploaded in "./crecerelle_results/data/dataset_name" *(optional)*. The object containing the gene expression matrix should have the prefix "adata_GE" whereas the one containing the transcript usage matrix should start with "adata_TU". Examples are:




*   adata_GE_{data_type}_preprocessed.h5ad
    *   e.g. adata_GE_full_preprocessed.h5ad
*   adata_TU_{data_type}_preprocessed.h5ad
    *   e.g. adata_TU_full_preprocessed.h5ad
*   adata_GE_{num_hvg}_{data_partition}.h5ad
    *   adata_GE_3000_train.h5ad
    *   adata_GE_3000_val.h5ad
    *   adata_GE_3000_test.h5ad

### Transcript usage data

In [ ]:
# Load transcript usage data
if settings["data_type"] != "full":
    adata_TU = ad.read_h5ad(path2data + "adata_TU_full_preprocessed.h5ad")
    intron_groups_by_name = adata_TU.var["intron_group"].values
    intron_groups = intron_names_2_integers(intron_groups_by_name)
    unique_intron_groups, _ = np.unique(intron_groups, return_index=True)
    num_intron_groups = len(unique_intron_groups)
    adata_TU.obs[settings["data_partition_key"]] = settings["data_type"]

    if settings["filter_cell_types"]:
        cell_types, cell_counts = np.unique(adata_TU.obs[settings["cell_type_key"]].to_numpy(), return_counts=True)
        filtered_cell_types = cell_types[np.argwhere(cell_counts >= settings["min_count_threshold"])].squeeze()

        # Filter adata_GE_train and adata_GE_val to filtered_cell_types
        adata_TU_filtered = adata_TU[adata_TU.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()

        adata_TU = adata_TU_filtered

else:
    # Load transcript usage training data
    adata_TU_train = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_train.h5ad")
    intron_groups_by_name = adata_TU_train.var["intron_group"].values
    intron_groups = intron_names_2_integers(intron_groups_by_name)
    unique_intron_groups, _ = np.unique(intron_groups, return_index=True)
    num_intron_groups = len(unique_intron_groups)
    adata_TU_train.obs[settings["data_partition_key"]] = "train"

    # Load transcript usage validation data
    adata_TU_val = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_val.h5ad")
    adata_TU_val.obs[settings["data_partition_key"]] = "val"

    # Load transcript usage test data
    adata_TU_test = ad.read_h5ad(path2data + "adata_TU_" + str(settings["num_hvg"]) + "_test.h5ad")
    adata_TU_test.obs[settings["data_partition_key"]] = "test"

    if settings["filter_cell_types"]:
        cell_types, cell_counts = np.unique(adata_TU_train.obs[settings["cell_type_key"]].to_numpy(), return_counts=True)
        filtered_cell_types = cell_types[np.argwhere(cell_counts >= settings["min_count_threshold"])].squeeze()

        # Filter adata_GE_train and adata_GE_val to filtered_cell_types
        adata_TU_train_filtered = adata_TU_train[adata_TU_train.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()
        adata_TU_val_filtered = adata_TU_val[adata_TU_val.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()
        adata_TU_test_filtered = adata_TU_test[adata_TU_test.obs[settings["cell_type_key"]].isin(filtered_cell_types)].copy()

        adata_TU_train = adata_TU_train_filtered
        adata_TU_val = adata_TU_val_filtered
        adata_TU_test = adata_TU_test_filtered

        print(f"Cell types with less than {settings["min_count_threshold"]} have been filtered out. \n")

    adata_TU = ad.concat([adata_TU_train, adata_TU_val, adata_TU_test], axis=0, join="outer", merge="same")

# @markdown **Optional: Filter cells in taxonomy level:** Per taxonomy level, filter out all cell types with less than min_cells (if not set to 0)
min_cells_per_tax_level = 50 # @param {type:"integer"}

if min_cells_per_tax_level > 0:
    # 1. Calculate the size of each cell type group within each tissue
    # transform('size') returns a Series with the same index as the original dataframe
    counts = adata_TU.obs.groupby([settings["cell_type_groups_key"], settings["cell_type_key"]])[settings["cell_type_key"]].transform('size')

    # 2. Create a boolean mask for cells to keep
    keep_mask = counts >= min_cells_per_tax_level

    # 3. Slice both AnnData objects in the tuple
    adata_TU_filtered = adata_TU[keep_mask, :].copy()

    adata_TU_filtered.obs[settings["cell_type_key"]] = adata_TU_filtered.obs[settings["cell_type_key"]].cat.remove_unused_categories()

    adata_TU = adata_TU_filtered.copy()
    adata_TU_train = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "train", :].copy()
    adata_TU_val = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "val", :].copy()
    adata_TU_test = adata_TU[adata_TU.obs[settings["data_partition_key"]] == "test", :].copy()

    print(f"Cell types with less than {min_cells_per_tax_level} cells per taxonomy level have been filtered out. \n")

print(f"Following Anndata object contains the transcript usage data of the data partition {settings["data_type"]}: \n {adata_TU} \n")
print(f"Following Anndata object contains the transcript usage training data: \n {adata_TU_train} \n")
print(f"Following Anndata object contains the transcript usage validation data: \n {adata_TU_val} \n")
print(f"Following Anndata object contains the transcript usage test data: \n {adata_TU_test} \n")


Cell types with less than 100 have been filtered out. 



/tmp/ipykernel_7375/2141861029.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = adata_TU.obs.groupby([settings["cell_type_groups_key"], settings["cell_type_key"]])[settings["cell_type_key"]].transform('size')
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Cell types with less than 50 cells per taxonomy level have been filtered out. 

Following Anndata object contains the transcript usage data of the data partition full: 
 AnnData object with n_obs × n_vars = 36256 × 2319
    obs: 'FACS.selection', 'age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'sex', 'subtissue', 'tissue', 'n_genes', 'n_counts', 'louvain', 'leiden', 'cell_type', 'plate_id', 'data_partition'
    var: 'chromosome', 'start', 'end', 'strand', 'annotated', 'gene_id_start', 'gene_id_end', 'n_genes', 'gene_id', 'gene_name', 'intron_group', 'intron_group_size', 'n_genes_per_intron_group'
    layers: 'counts', 'psi' 

Following Anndata object contains the transcript usage training data: 
 AnnData object with n_obs × n_vars = 29028 × 2319
    obs: 'FACS.selection', 'age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'sex', 'subtissue', 'tissue', 'n_genes', 'n_counts', 'louvain', 'le

## Dataset and dataloader

In [ ]:
# @markdown **Dataset and dataloader:** Create the training and validation dataset of transcript usage and corresponding dataloader for tuVI

# @markdown **Batch size:** Choose the batch size for training. It is recommended to use batch sizes that are powers of 2 and greater than 100
batch_size = 256 # @param {type:"number"}

# Create training dataset and dataloader
cell_intron_counts_train = torch.from_numpy(adata_TU_train.layers["counts"].toarray())
#cell_intron_levels = torch.from_numpy(adata_TU_train.layers["psi"])
cell_intron_levels_train = torch.from_numpy(adata_TU_train.X.toarray())

transform_intron_counts = None # Previously "log" but incorrect
transform_intron_levels = None
transform_ontology = "one-hot"

train_dataset = TranscriptUsageDataset(
    cell_intron_counts=cell_intron_counts_train,
    cell_intron_levels=cell_intron_levels_train,
    ontology=adata_TU_train.obs[settings["cell_type_key"]].to_numpy(),
    tissue=adata_TU_train.obs[settings["cell_type_groups_key"]].to_numpy(),
    cell_types= cell_types,
    transform_intron_counts=transform_intron_counts,
    transform_intron_levels=transform_intron_levels,
    transform_ontology=transform_ontology
)

# Dataloader
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)

print(f"The dataloader for the training dataset is set up.\n")

# Create validation dataset and dataloader
cell_intron_counts_val = torch.from_numpy(adata_TU_val.layers["counts"].toarray())
#cell_intron_levels = torch.from_numpy(adata_TU_val.layers["psi"])
cell_intron_levels_val = torch.from_numpy(adata_TU_val.X.toarray())

transform_intron_counts = None # Previously "log" but incorrect
transform_intron_levels = None
transform_ontology = "one-hot"

val_dataset = TranscriptUsageDataset(
    cell_intron_counts=cell_intron_counts_val,
    cell_intron_levels=cell_intron_levels_val,
    ontology=adata_TU_val.obs[settings["cell_type_key"]].to_numpy(),
    tissue=adata_TU_val.obs[settings["cell_type_groups_key"]].to_numpy(),
    cell_types= cell_types,
    transform_intron_counts=transform_intron_counts,
    transform_intron_levels=transform_intron_levels,
    transform_ontology=transform_ontology
)

# Dataloader
val_loader = DataLoader(val_dataset, batch_size, shuffle=True)

print(f"The dataloader for the validation dataset is set up.")


The dataloader for the training dataset is set up.

The dataloader for the validation dataset is set up.


# Transcript usage Variational Inference (tuVI)

In [ ]:
# @markdown **Dimension of latent space:** Define the dimensionality of the cell embeddings
latent_dim = 10 # @param {type:"integer"}

# @markdown **Beta parameter:** Choose the value for the beta parameter
beta = 1.0 # @param {type:"number"}

# @markdown **Architecture of neural networks:** choose the number of hidden layers as well as the number of hidden units per layer for the encoding and decoding neural networks
num_hidden_layers = 2 # @param {type:"integer"}
num_hidden_units = 256 # @param {type:"integer"}

# @markdown **Dropout rate:** If dropout shall be applied, set the dropout rate to 0.0 > p > 1.0. Otherwise, choose 0.0.
dropout_rate = 0.1 # @param {type:"number"}
scaling_factor = True # @param {type:"boolean"}
device = "cuda" # @param ["cuda", "cpu"]

# Convert beta to string
beta_str = f"{beta:.1f}"
beta_str = beta_str.replace(".", "")
while len(beta_str) < 3:
    beta_str = "0" + beta_str

## tuVI with Dirichlet-Multinomial observation model

In [ ]:
from crecerelle.models import DMTranscriptUsageVAE

model = DMTranscriptUsageVAE(
    input_dim=train_dataset.num_introns,
    num_intron_groups=num_intron_groups,
    intron_groups = intron_groups,
    latent_dim=latent_dim,
    beta=beta,
    num_hidden_layers=num_hidden_layers,
    num_hidden_units=num_hidden_units,
    dropout_rate=dropout_rate,
    device=device,
    scaling_factor=False # by default
)

# @markdown **Observation model of tuVI:** Dirichlet-Multinomial (DM) likelihood
likelihood_type = "DM"
count_data_included = True

# @markdown **Random seed:** select the random seed
seed = 7 # @param {type:"integer"} 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
torch.manual_seed(seed)
np.random.seed(seed)

# Nomenclature transcriptUsage + likelihood + VAE + seed + beta
model_name = "tuVI_" + likelihood_type + "_" + str(seed) + "_Beta_" + beta_str
loss_type = "VariationalELBO"

print(f"The given model specification for tuVI yields: \n {model}")

The given model specification for tuVI yields: 
 DMTranscriptUsageVAE(
  (encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): ModuleList(
        (0): Sequential(
          (linear): Linear(in_features=2319, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): Sequential(
          (linear): Linear(in_features=256, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=256, out_features=10, bias=True)
    (var_encoder): Linear(in_features=256, out_features=10, bias=True)
  )
  (decoder): IntronsDecoder(
    (non_linear_layers): FCLayers(
      (fc_layers): ModuleList(
    

/usr/local/lib/python3.13/dist-packages/crecerelle/models.py:1367: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  torch.sparse_coo_tensor(


## tuVI with zero-and-N-inflated Dirichlet-Multinomial (ZANIDM) observation model

In [ ]:
from crecerelle.models import ZANIDMTranscriptUsageVAE

# @markdown **Observation model of tuVI:** Zero-and-N-inflated Dirichlet-Multinomial likelihood
model = ZANIDMTranscriptUsageVAE(
    input_dim=train_dataset.num_introns,  # number of introns
    num_intron_groups=num_intron_groups,
    intron_groups=intron_groups,
    latent_dim=latent_dim,
    beta=beta,
    num_hidden_layers=num_hidden_layers,
    num_hidden_units=num_hidden_units,
    dropout_rate=dropout_rate,
    device=device
)

# @markdown **Observation model of tuVI:** Zero-and-N-inflated Dirichlet-Multinomial (ZANIDM) likelihood
likelihood_type = "ZANIDM"
count_data_included = True

# @markdown **Random seed:** select the random seed
seed = 7 # @param {type:"integer"} 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
torch.manual_seed(seed)
np.random.seed(seed)

# Nomenclature transcriptUsage + likelihood + VAE + seed + beta
model_name = "tuVI_" + likelihood_type + "_" + str(seed) + "_Beta_" + beta_str
loss_type = "VariationalELBO"

print(f"The given model specification for tuVI yields: \n {model}")

The given model specification for tuVI yields: 
 ZANIDMTranscriptUsageVAE(
  (encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): ModuleList(
        (0): Sequential(
          (linear): Linear(in_features=2319, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): Sequential(
          (linear): Linear(in_features=256, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=256, out_features=10, bias=True)
    (var_encoder): Linear(in_features=256, out_features=10, bias=True)
  )
  (decoder): ZANIDMTranscriptUsageDecoder(
    (decoding_layers): FCLayers(
      (fc_layers): 

## tuVI with ZANIDM heuristic (ZIDM)

In [ ]:
from crecerelle.models import ZIDMTranscriptUsageVAE

# @markdown **Observation model of tuVI:** Zero-inflated Dirichlet-Multinomial likelihood
model = ZIDMTranscriptUsageVAE(
    input_dim=train_dataset.num_introns,  # number of introns
    num_intron_groups=num_intron_groups,
    intron_groups=intron_groups,
    latent_dim=latent_dim,
    beta=beta,
    num_hidden_layers=num_hidden_layers,
    num_hidden_units=num_hidden_units,
    dropout_rate=dropout_rate,
    device=device
)

# @markdown **Observation model of tuVI:** ZANIDM heuristic (ZIDM)
likelihood_type = "ZIDM"
count_data_included = True

# @markdown **Random seed:** select the random seed
seed = 9 # @param {type:"integer"} 0, 1, 2, 3, 4, 5, 6, 7, 8, 9
torch.manual_seed(seed)
np.random.seed(seed)

# Nomenclature transcriptUsage + likelihood + VAE + seed + beta
model_name = "tuVI_"+ likelihood_type + "_" + str(seed) + "_Beta_" + beta_str
loss_type = "VariationalELBO"

print(f"The given model specification for tuVI yields: \n {model}")

The given model specification for tuVI yields: 
 ZIDMTranscriptUsageVAE(
  (encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): ModuleList(
        (0): Sequential(
          (linear): Linear(in_features=2319, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): Sequential(
          (linear): Linear(in_features=256, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=256, out_features=10, bias=True)
    (var_encoder): Linear(in_features=256, out_features=10, bias=True)
  )
  (decoder): ZIDMTranscriptUsageDecoder(
    (decoding_layers): FCLayers(
      (fc_layers): Modu

# Training

In [ ]:
# @markdown **Weight-Initialization:** The parameters of the VAE are initialised using Kaiminig-He
def init_weights_he(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')

        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)

        print(f"{m} is initialised with Kaiming-He")

model.apply(init_weights_he)

Linear(in_features=2319, out_features=256, bias=True) is initialised with Kaiming-He
Linear(in_features=256, out_features=256, bias=True) is initialised with Kaiming-He
Linear(in_features=256, out_features=10, bias=True) is initialised with Kaiming-He
Linear(in_features=256, out_features=10, bias=True) is initialised with Kaiming-He
Linear(in_features=10, out_features=128, bias=True) is initialised with Kaiming-He
Linear(in_features=128, out_features=256, bias=True) is initialised with Kaiming-He
Linear(in_features=256, out_features=2319, bias=True) is initialised with Kaiming-He
Linear(in_features=256, out_features=2319, bias=True) is initialised with Kaiming-He


ZIDMTranscriptUsageVAE(
  (encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): ModuleList(
        (0): Sequential(
          (linear): Linear(in_features=2319, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): Sequential(
          (linear): Linear(in_features=256, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=256, out_features=10, bias=True)
    (var_encoder): Linear(in_features=256, out_features=10, bias=True)
  )
  (decoder): ZIDMTranscriptUsageDecoder(
    (decoding_layers): FCLayers(
      (fc_layers): ModuleList(
        (0): Sequential(
          (linea

In [ ]:
# @markdown **GPU computation:** Push model to GPU if available
if torch.cuda.is_available():
    model.to(device)
else:
    model.to(device)

print(f"The given model specification for tuVI yields: \n {model}")

The given model specification for tuVI yields: 
 ZIDMTranscriptUsageVAE(
  (encoder): Encoder(
    (encoder): FCLayers(
      (fc_layers): ModuleList(
        (0): Sequential(
          (linear): Linear(in_features=2319, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): Sequential(
          (linear): Linear(in_features=256, out_features=256, bias=True)
          (batchnorm): BatchNorm1d(256, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (mean_encoder): Linear(in_features=256, out_features=10, bias=True)
    (var_encoder): Linear(in_features=256, out_features=10, bias=True)
  )
  (decoder): ZIDMTranscriptUsageDecoder(
    (decoding_layers): FCLayers(
      (fc_layers): Modu

In [ ]:
# @markdown **Number of epochs:** Choose the number of epochs to train the VAE
num_epochs = 300 # @param {type:"integer"}
# @markdown **Early stopping:** If early stopping shall be used, define the number of epochs without improvement after which the training is stopped
patience = 20 # @param {type:"integer"}
# @markdown **Initial learning rate:** Define the initial learning rate for the optimisation
init_lr = 0.001 # @param {type:"number"}
# @markdown **Learning rate scheduler:** For advanced optimisation, choose one of the learning rate schedulers. Otherwise, select None.
lr_scheduler = None # @param ["None", "CosineAnnealingLR", "ReduceLROnPlateau"] {type:"raw"}

optimizer = torch.optim.AdamW(model.parameters(), lr=init_lr)

# Apply gradient clipping
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Enable anomaly detection to help pinpoint the source of the RuntimeError
torch.autograd.set_detect_anomaly(True)

train_VAE(
    model,
    optimizer,
    (train_loader, val_loader),
    lr_scheduler,
    num_epochs,
    model_name,
    settings["dataset_name"] + "_" + str(settings["num_hvg"]),
    loss_type,
    count_data_included,
    patience
)

EPOCH 1:
   batch 10 loss: 895.4150756835937
        NLL: 886.7071472167969, KLD: 8.707926273345947
   batch 20 loss: 655.3339904785156
        NLL: 644.7369506835937, KLD: 10.597034358978272


KeyboardInterrupt: 

In [ ]:
import os
import glob
import shutil

# @markdown **Move best models:** Move the best tuVI models trained from ./models to the destination directory

# Source and destination directories
src_dir = "./models"
dst_dir = "./models/tuVI" # @param

# Make sure destination exists
os.makedirs(dst_dir, exist_ok=True)

# Match any epoch_num using *
pattern = (
    f'{settings["dataset_name"]}_'
    f'{settings["num_hvg"]}_'
    f'tuVI_'
    f'{likelihood_type}_'
    f'{seed}_'
    f'Beta_{beta_str}_'
    f'epochs_*_checkpoint.pth'
)

matches = glob.glob(os.path.join(src_dir, pattern))

if len(matches) == 0:
    raise FileNotFoundError(f"No checkpoint found matching: {pattern}")

for src_path in matches:
    dst_path = os.path.join(dst_dir, os.path.basename(src_path))

    shutil.move(src_path, dst_path)

    print(f"Moved:\n{src_path}\n→ {dst_path}")

tuVI-ZIDM needs 26.5 seconds per epoch
tuVI-ZANIDM needs 3900 seconds per epoch